In [1]:
import json, yaml, os
import numpy as np
import cv2
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import random
from microlane.evalutation.lane_eval import LaneEval

In [2]:
# First Load the Configuation file
with open("configs/config.yaml", "r") as file:
    config = yaml.safe_load(file)

In [3]:
gt_json_path = config['data']['datasets']['tusimple']['annotation_file']

prediction_json_path = "results/testing/2026_04_25__13_13_32_batch_pipeline_testing_with_no_augmentation_with_ufld_model/prediction.json"

In [4]:
output_dir = os.path.dirname(prediction_json_path)

output_path = os.path.join(output_dir, 'evaluation.json')

In [5]:
## Loading the Json files respectively

with open(prediction_json_path, 'r') as f:
    content = f.read().strip()
    try:
        # Try loading as full JSON (list of dicts)
        json_pred = json.loads(content)
        if not isinstance(json_pred, list):
            raise Exception("Prediction JSON must be a list of objects.")
    except Exception:
        # Fallback: assume JSON lines format
        json_pred = [json.loads(line) for line in content.splitlines() if line.strip()]
    
json_gt = [json.loads(line) for line in open(gt_json_path)]

In [6]:
results = []

for pred, gt in (zip(json_pred, json_gt)):
    
    pred_lanes = pred['lanes']
    run_time = pred['run_time']
    gt_lanes = gt['lanes']
    y_samples = gt['h_samples']
    raw_file = gt['raw_file']
    
    accuracy, fp, fn = LaneEval.bench(pred_lanes, gt_lanes, y_samples, run_time)
    
    results.append({
        'raw_file': raw_file,
        
        'accuracy': accuracy,
        
        'fp': fp,
        
        'fn': fn,
        
        'run_time': run_time
    })

with open(output_path, 'w') as f:
    json.dump(results, f, indent=4)